In [12]:
import pandas as pd
import altair as alt
import re
import numpy as np

alt.data_transformers.disable_max_rows()


# import nltk
# from nltk.tokenize import sent_tokenize
# nltk.download('punkt', quiet=True)
# def count_sentences(text):
#     return len(sent_tokenize(text))

def count_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    sentences = [s for s in sentences if s.strip()]
    return max(len(sentences), 1)


df_episodes = pd.read_csv(
    "C:\\Users\\20203666\\Documents\\GitHub\\data-vis-simpsons\\assignment2\\data\\raw_data\\simpsons_episodes.csv")
df_lines = pd.read_csv(
    "C:\\Users\\20203666\\Documents\\GitHub\\data-vis-simpsons\\assignment2\\data\\raw_data\\simpsons_script_lines.csv")
df_lines = df_lines.merge(
    df_episodes[['id', 'number_in_season', 'season']],
    left_on='episode_id',
    right_on='id',
    how='left'
)
df_lines['word_count'] = pd.to_numeric(df_lines['word_count'], errors='coerce')
df_lines = df_lines[df_lines['word_count'] <= 150]
df_lines = df_lines[df_lines['season']!=26]
df_lines.dropna(subset=['raw_character_text', 'normalized_text', 'word_count'], inplace=True)
df_lines.drop(['id_x', 'id_y'], axis=1, inplace=True)
df_lines['lower_characters'] = df_lines['raw_character_text'].str.lower()
df_lines['timestamp_in_ms'] = pd.to_numeric(df_lines['timestamp_in_ms'])
df_lines['timestamp_in_min'] = (df_lines['timestamp_in_ms'] // 60000) + 1
df_lines['sentence_count'] = df_lines['spoken_words'].apply(count_sentences)
df_lines

C:\Users\20203666\AppData\Local\Temp\ipykernel_6988\761321532.py:23: DtypeWarning: Columns (4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lines = pd.read_csv(


,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count,number_in_season,season,lower_characters,timestamp_in_min,sentence_count
0,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31.0,19,2,miss hoover,15,2
1,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3.0,19,2,lisa simpson,15,2
2,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,True,464.0,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22.0,19,2,miss hoover,15,4
3,32,212,Lisa Simpson: That life is worth living.,864000,True,9.0,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5.0,19,2,lisa simpson,15,1
4,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,True,40.0,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33.0,19,2,edna krabappel-flanders,15,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158266,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2.0,19,2,miss hoover,14,1
158267,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10.0,19,2,miss hoover,14,1
158268,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1.0,19,2,miss hoover,15,1
158269,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6.0,19,2,ralph wiggum,15,1


In [13]:

top_characters = (
    df_lines.groupby('raw_character_text')['word_count']
    .sum()
    .sort_values(ascending=False)
    .head(13)
    .index
    .tolist()
)
top_characters
name_characters = ['Homer Simpson',
                   'Marge Simpson',
                   'Bart Simpson',
                   'Lisa Simpson',
                   'C. Montgomery Burns',
                   'Moe Szyslak',
                   'Seymour Skinner',
                   'Ned Flanders',
                   'Krusty the Clown',
                   'Chief Wiggum',
                   'Grampa Simpson',
                   'Kent Brockman',
                   'Milhouse Van Houten',
                   'Apu Nahasapeemapetilon',
                   'Lenny Leonard',
                   'Waylon Smithers',
                   'Nelson Muntz',
                   'Dr. Julius Hibbert',
                   'Carl Carlson',
                   'Edna Krabappel-Flanders'
                   ]

short_name_characters = ['Homer',
                         'Marge',
                         'Bart',
                         'Lisa',
                         'Burns',
                         'Moe',
                         'Skinner',
                         'Flanders',
                         'Krusty',
                         'Wiggum',
                         'Grampa',
                         'Kent',
                         'Milhouse',
                         'Apu',
                         'Lenny',
                         'Smithers',
                         'Nelson',
                         'Hibbert',
                         'Carl',
                         'Edna'
                         ]

lower_name_characters = [n.lower() for n in short_name_characters]
pattern = '|'.join(lower_name_characters)
df_names = df_lines[(df_lines['lower_characters'].str.contains(pattern, na=False)) & ~(
    df_lines['raw_character_text'].isin(name_characters))]
match_names = ["Bart's Head", 'ADVISORS & SMITHERS',
               'BETTY & MOE'
               "HOMER (CONT'D",
               'Skinner Body', 'BART & LISA', 'Homer-Ape', 'Baby Lisa', 'CHIF WIGGUM',
               "HOMER'S THOUGHT",
               'Skinner Zombie', 'Zombie Krusty', 'Zombie Flanders', 'Lisa Snail',
               'Homer', "Homer's Spirit", 'Bart Anchor', "Homer's Thought Bubble",
               "Homer's Bloody Skull", "Apu's Brain", "Homer's Conscience",
               "Burns' Grandfather", "Bart's Thoughts", "Lisa's Thoughts",
               "Homer's Brain", 'Evil Homer',
               '2-Year-Old Lisa', '7-YEAR OLD BURNS', 'Pieces Of Homer', "Flanders' Inner Child", "Homer's Inner Child",
               "Moe's Inner Child", 'Triple Milhouse', 'Jerry Lewis Bart',
               'Cherub Lenny', 'Cherub Carl', 'Baby Burns', 'Teenage Burns',
               'Marge', 'Lisa',
               'Baby Bart',
               'Lunchlady Bart', "Grampa's Brain", 'Nelson Apparition', '7-Year-Old Homer',
               '7-YEAR-OLD-HOMER', "Bart's Fist",
               'Adult Nelson', 'Imaginary Burns #1', 'Imaginary Burns #2',
               'Imaginary Burns #3', 'Ten Imaginary Burnses', "Smithers's Brain",
               'Pre-Teen Hibbert Kid', 'Little Hibbert Girl',
               'Teenage Hibbert Boy', 'NELSON/HIS SOUL', 'Thistlewick Flanders', "Homer's Mouth",
               'Student Wiggum', 'Phony Homer', 'Phony Marge', 'Young Apu', 'Apu II', 'Milhouse',
               'Simulated Homer', 'Bernice Hibbert', "Marge's Dummy",
               'Flashback Homer', 'Flashback Marge', "Homer's Recording",
               "Milhouse's Thoughts", "Nelson's Brain", "SMITHERS'", "Smithers's Echo",
               "Mr. Burns's Thoughts", '1986 Krusty', 'Young Flanders', 'Sgt. Skinner', 'Krusty', 'Mutant Dr. Hibbert',
               'Mutant Moe', 'Mutant Skinner', 'Mutant Burns', 'Mutant Lenny',
               'Mutant Wiggum', 'MUTANT FLANDERS', 'Constable Wiggum',
               'Goodman Skinner', 'Goodman Flanders', 'Goodman Moe',
               'Goodman Lenny', 'Teenage Smithers', 'MIDDLE-AGE GRAMPA',
               'Teenage Lenny', "Homer's Stomach", "Homer's Feet",
               "Homer's Voice", 'GHOSTLY HOMER', 'Thought Bubble Moe',
               'Thought Bubble Bart', "HOMER'S VOICE MAIL", 'Cardinal Flanders',
               'President Lenny', 'BART BART', 'Middle-aged Grampa',
               '3-Year-Old Homer', 'Bad Flanders', 'DETECTIVE HOMER SIMPSON',
               'Pharaoh Skinner', 'Moses/milhouse', 'Methuselah Grampa',
               'Goliath/nelson', 'Homer Effigy',
               'LISA JUNIOR', '20-ISH MOE', 'Hologram Nelson', 'Old Krusty',
               'Latin Milhouse', 'LISABELLA', 'Thought Bubble Homer',
               "LISA'S LAWYER", 'Willie Nelson', 'Ghost Homer',
               'Virtual Wiggum', 'Fake Homer', 'APU+', 'Young Skinner',
               'Camel Lisa', 'Harem Girl Bart', 'SIX-YEAR-OLD HOMER',
               'Sourdough Moe', 'Sheriff Wiggum', "Bart's Prince",
               'KRUSTY FACE', 'MR. BURNS BRAIN', 'Young Lenny', 'Young Carl',
               'Young Moe', "Grampa's Recorded Voice",
               'Lenny Pig', 'Hamlet Bart', 'Guilden-Lenny',
               'Rosen-Carl', 'Present-Day Homer', 'Cartoon Bart', 'Dream Homer',
               'Homer Devil', 'Homer Angel', 'Homer Double',
               'TRACEY ULLMAN SHOW HOMER', 'HOMER DOUBLES', 'Future Homer',
               'Cow Flanders', 'Pig Wiggum', 'Owl Lisa', 'Fox Burns', 'LISA OWL',
               'Spider Bart', 'Walrus Homer', 'Panther Marge', 'Drunk Homer',
               'HOMER THOUGHTS', 'SIX-YEAR-OLD GRAMPA', "Skinner's Thoughts",
               "Mrs. Skinner's Thoughts", "FLANDERS' VOICE", 'Sheriff Lisa',
               "Bart's Voice", '60-Year-Old Nelson', 'Real Lisa', 'Hibbert Boy',
               "Flanders's Thoughts", 'MOE HOWARD', 'Animated Skinner',
               'PUZZLE LENNY', 'Young Marge', 'Young Dr. Hibbert',
               "Homer's Thoughts", "HOMER-ISH WOMAN'S VOICE",
               'Knockahomer', 'Bartleby', 'Prince Bart', 'Advisor Moe',
               'Chief Homer', 'Chancellor Smithers', "Marge's Thoughts",
               'Skinner On Plaque', 'CYBORG SPIDER MRS. SKINNER',
               'PRINCIPAL SKINNER CT', '10 Years Younger Homer',
               '10 Years Younger Marge', '10-Year-Old Homer', '10-Year-Old Lenny',
               '10-Year-Old Carl', 'TEN-YEAR-OLD MARGE', "Homer's Mind",
               "Marge's Mind", 'Inspector Wiggum',
               'Ebenezer Burns', '2nd Homer', "Marge's Face", 'Teenage Moe',
               "Lisa's Reflection", "Homer's Muzzle", 'MOE-BIRD',
               "Moe's Thoughts", 'Mr. Burns Logo', 'Bart on Tape', 'Homers',
               'SIX-YEAR-OLD-BART', 'Muscular Homer', 'Teenage Nelson',
               'Teenage Milhouse', 'Teenage Lisa', 'Robo-Wiggum', 'MOE-CLONE',
               'Spider Moe', '50-ish Lisa', '50-ish Milhouse', 'President Homer',
               'Sergeant Skinner', 'Bart Spider', 'Milhouse Slug',
               'Grampa Gorilla', 'Dracula Hibbert', 'Einstein Lisa',
               "Homer's Head", 'Werewolf Bart', 'Apu Robot', 'Moe Pacifier',
               'Lenny Pacifier', 'Old Marge', 'Thought Bubble Marge',
               'Younger Milhouse', 'Hibbert Wise Man', 'Skinner Wise Man',
               'Shepherd Carl', 'Shepherd Lenny', 'WISE MAN HIBBERT',
               'Young Burns', 'Toddler Homer', '40-ish Grampa', 'Fun Homer',
               'Serious Homer', 'Bart-Jack', 'Bart-waitress', 'Bart Head',
               'Moe Recording', "Mr. Burns's Reflection", 'Beefeater Lenny',
               'Beefeater Carl', 'Moezekiel', 'Pilgrim Smithers', 'King Homer',
               'Queen Marge', 'Capt. Burns', 'CHIEF PETTY OFFICER WIGGUM',
               'C.P.O. Wiggum', 'Assistant G.K. Skinner', 'Teenage Marge',
               'Little Moe Szyslak', '50-Foot Lenny', 'Invisible Carl',
               '7-Year-Old Marge', 'Gendarme Wiggum', 'Bart-man', 'Poison Lenny',
               'Elf Marge', '11-Year-Old Moe', '8-Year-Old Lenny',
               '8-Year-Old Carl', '8-Year-Old Homer', '8-Year-Old Wiggum',
               '16-Year-Old Wiggum', '24-Year-Old Wiggum', '32-Year-Old Wiggum',
               '24-Year-Old Homer', '32-Year-Old Homer', '8-Year-Old Marge',
               '24-Year-Old Marge', '2-Year-Old Bart', 'VENDOR APU',
               'Bird Skinner', 'Troll Moe', 'Wench Milhouse', "Nelson's Head",
               'Old Bart', 'Old Milhouse', 'Ghost Marge',
               'Memory Wiggum', 'Memory Marge', 'Memory Homer', 'Memory Lisa',
               'Memory Bart', '20-Year-Old Homer', 'Young Grampa', 'Young Homer',
               'Moe Dog', 'Lisa Puppy', 'Bart Puppy', 'Prisoner Lisa',
               'Smoke Lisa', 'Little Homer', '3-Year-Old Lisa', '5-Year-Old Bart',
               'Thought Bubble Lenny', 'Teenage Carl',
               'Young Smithers', "HOMER'S HAND", 'German Krusty',
               'Wiggum Smiley Face', 'Carl #2', 'Carl #1', 'Homer Snowman',
               'Adult Lisa',
               'Colonel Burns', 'Thought Bubble Grampa',
               'Thought Bubble Lisa', 'MARGE 40-YEAR-OLD',
               'Mr. Burns Heads', 'Devil Moe',
               'Avatar Bart', 'THOUGHT-BUBBLE FLANDERS', '1970s Grampa',
               "Lenny's Face", "Skinner's Face", "Moe's Face", 'Avatar Milhouse',
               'Adult Bart', 'Elderly Principal Skinner', 'Adult Milhouse',
               'Aged Burns', 'Aged Krusty', 'Lenny', 'Carl', 'Bartholomé',
               'Archbishop Smithers', 'French Wiggum', 'Viking Lenny',
               'Viking Homer', 'Christian Homer', 'King Nelson',
               'Thought Bubble Apu', '"Shorts" Homer', '"Shorts" Bart',
               '"Shorts" Lisa', '"Shorts" Marge',
               "FLANDERS' MOUTH AND MOUSTACHE", 'Elderly Homer', 'Elderly Marge',
               '1-Year-Old Homer', '80-Year-Old Bart', 'Elderly Nelson',
               "Homer's Reflection", 'Older Flanders', 'Little Marge',
               'Corrupt Pope Homer', 'Fop Homer', 'Egyptian Slave Homer',
               'Thought Bubble Dr. Hibbert', 'Angel Skinner', 'DUFFMAN VOICE MILHOUSE',
               'Stanley Kowalski Milhouse', '1-Year-Old Bart',
               '1-Year-Old Nelson', '800-Pound Homer', "Kent Brockman's Thoughts",
               'MR. BURNS.',
               'Rastafarian Krusty', 'Somber Irish Krusty', 'IRISH KRUSTY',
               'Mexican Krusty', 'Botswana Krusty', 'Bart Snail',
               'Animated Krusty', 'MARGE 7',
               'ZOMBIE MILHOUSE', '12-Year-Old Homer',
               'Super Milhouse', "KRUSTY'S HEAD", 'TEENAGED KRUSTY',
               'MIDDLE-AGED KRUSTY', 'Krustys', 'Caveman Homer', 'ORIGINAL HOMER', 'ORIGINAL BART',
               'ORIGINAL MARGE', 'REGULAR GHOST MARGE', 'REGULAR GHOST LISA',
               'REGULAR GHOST HOMER', 'ORIGINAL GHOST MARGE',
               'REGULAR GHOST MARGE/ ORIGINAL GHOST MARGE', 'ORIGINAL GHOST BART',
               'ORIGINAL GHOST HOMER', 'CGI HOMER', 'ORIGINAL GHOST LISA',
               'MARGE *', 'NELSON COMET', 'HOMER)',
               "SMITHERS' THOUGHTS", 'AGED MOE', 'Teenage Homer', 'Young Krusty',
               'Teenage Bart', 'Adult "Bart"', 'Kirk Voice Milhouse', 'Homer The Thief', 'KENT']
replace_dict = dict(zip(lower_name_characters, name_characters))

match_set = set(match_names)


def process_name(name):
    if name in match_set:
        for lower, real in replace_dict.items():
            if lower in name.lower():
                return real
    return name


df_lines['character_names'] = df_lines['raw_character_text'].apply(process_name)

df_final = df_lines[
    ['character_names', 'episode_id', 'season', 'number_in_season', 'timestamp_in_min', 'word_count', 'sentence_count']]
df_final = df_final[df_final['character_names'].isin(top_characters)]


def fill_missing_combinations(df, entity_cols, grid_cols, value_col, agg_func='sum', fill_value=0):
    """
    Groups a dataframe and ensures every entity has a row for every valid grid combination,
    filling missing data with a specified value.
    """
    if isinstance(entity_cols, str): entity_cols = [entity_cols]
    if isinstance(grid_cols, str): grid_cols = [grid_cols]
    all_group_cols = entity_cols + grid_cols
    grouped_df = df.groupby(all_group_cols)[value_col].agg(agg_func).reset_index()
    unique_entities = df[entity_cols].drop_duplicates()
    if len(grid_cols) > 2:
        parent_cols = grid_cols[:-1]
        target_col = grid_cols[-1]
        max_df = df.groupby(parent_cols)[target_col].max().reset_index()
        max_df[target_col] = max_df[target_col].apply(lambda x: list(range(1, int(x) + 1)))
        valid_grid_points = max_df.explode(target_col).reset_index(drop=True)
        valid_grid_points = valid_grid_points[grid_cols]
    else:
        valid_grid_points = df[grid_cols].drop_duplicates()
    master_grid = unique_entities.merge(valid_grid_points, how='cross')
    complete_df = master_grid.merge(grouped_df, on=all_group_cols, how='left')
    complete_df[value_col] = complete_df[value_col].fillna(fill_value)
    complete_df = complete_df.sort_values(by=all_group_cols).reset_index(drop=True)

    return complete_df


df_q1 = df_final.groupby(['character_names', 'episode_id'])[['word_count', 'sentence_count']].sum().reset_index()
df_q2 = fill_missing_combinations(
    df=df_final,
    entity_cols='character_names',
    grid_cols=['season'],
    value_col='word_count'
)
df_q3 = fill_missing_combinations(
    df=df_final,
    entity_cols='character_names',
    grid_cols=['season', 'number_in_season'],
    value_col='word_count'
)
df_q4 = fill_missing_combinations(
    df=df_final,
    entity_cols='character_names',
    grid_cols=['season', 'number_in_season', 'timestamp_in_min'],
    value_col='word_count'
)
df_q5 = df_final.groupby(['character_names', 'episode_id'])['sentence_count'].sum().reset_index()
# Save each DataFrame to its own CSV file
df_q1.to_csv('../data/clean_data/df_q1.csv', index=False)
df_q2.to_csv('../data/clean_data/df_q2.csv', index=False)
df_q3.to_csv('../data/clean_data/df_q3.csv', index=False)
df_q4.to_csv('../data/clean_data/df_q4.csv', index=False)
df_q5.to_csv('../data/clean_data/df_q5.csv', index=False)